In [1]:
from pathlib import Path

import polars as pl
import polars_h3 as plh3
import yaml

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [62]:
RES = 10
AGGREGATION_BY = "device"  # trip | device
THRESHOLD = 3

In [63]:
path = f"{config['dirs']['output']}/{config['city']}"

In [64]:
hw = pl.read_parquet(f"{path}/homes/home_work_h3_{RES}.parquet")
hw = hw.rename({f"h3_{RES}_home": "home", f"h3_{RES}_work": "work"})
hw = hw.drop_nulls()

hw = hw.filter((pl.col("count_home") > THRESHOLD) & (pl.col("count_work") > THRESHOLD))

In [65]:
def determine_visitation_network(home_work: pl.DataFrame, res: int) -> pl.DataFrame:
    result = pl.DataFrame()
    for i in range(256):
        temp = pl.read_parquet(
            f"{path}/hexagon_network/{res}/connections_{i:02x}.parquet"
        )

        m = temp.join(hw, on="device_id")
        from_home = m.filter(
            (pl.col("source") == pl.col("home")) & (pl.col("target") != pl.col("work"))
        )
        from_work = m.filter(
            (pl.col("source") == pl.col("work")) & (pl.col("target") != pl.col("home"))
        )
        result = pl.concat([result, from_home])
        result = pl.concat([result, from_work])
    return result.group_by(["device_id", "source", "target"]).agg(pl.sum("weight"))

In [66]:
visitations = determine_visitation_network(hw, res=RES)

In [67]:
aggregated = (
    visitations.join(hw, on="device_id")
    .group_by(["home", "work", "source", "target"])
    .agg(pl.sum("weight"))
)
if AGGREGATION_BY == "device":
    aggregated = (
        visitations.join(hw, on="device_id")
        .group_by(["home", "work", "source", "target"])
        .agg(pl.col("device_id").unique().len().alias("weight"))
    )

aggregated = aggregated.sort(["home", "work", "source", "target"])

In [68]:
Path(f"{path}/hexagon_network/{RES}/commuting_aware/{AGGREGATION_BY}").mkdir(
    exist_ok=True, parents=True
)
aggregated.write_csv(
    f"{path}/hexagon_network/{RES}/commuting_aware/{AGGREGATION_BY}/{RES}.csv"
)
aggregated.write_parquet(
    f"{path}/hexagon_network/{RES}/commuting_aware/{AGGREGATION_BY}/{RES}.parquet"
)